# Smoke test: BGE reranker fine-tuning

This is a plumbing test only: one query, one positive passage, five negative passages, and one optimizer step. It verifies model download, tokenization, CUDA forward/backward, FlagEmbedding training, and checkpoint saving without encoding the legal corpus. It is not an experiment and produces no reportable metric.

In [ ]:
from kaggle_secrets import UserSecretsClient
from pathlib import Path
import os

token = UserSecretsClient().get_secret('HF_TOKEN')
if not token:
    raise RuntimeError('Kaggle Secret HF_TOKEN is missing')
os.environ['HF_TOKEN'] = token
WORK = Path('/kaggle/working/reranker_smoke_test')
WORK.mkdir(parents=True, exist_ok=True)

In [ ]:
!python -m pip install 'FlagEmbedding==1.4.2'

In [ ]:
import json
import torch

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable')
print('GPU:', torch.cuda.get_device_name(0))
train_path = WORK / 'one_query.jsonl'
record = {
    'query': 'Phạm nhân không biết chữ có được học văn hóa để xóa mù chữ không?',
    'pos': ['Phạm nhân chưa biết chữ được tạo điều kiện học văn hóa để xóa mù chữ.'],
    'neg': [
        'Quy định về mở và sử dụng tài khoản vốn bằng ngoại tệ.',
        'Điều kiện cấp giấy phép xây dựng đối với nhà ở riêng lẻ.',
        'Thủ tục đăng ký thay đổi nội dung đăng ký doanh nghiệp.',
        'Quy định về thuế đối với hoạt động xuất khẩu hàng hóa.',
        'Trình tự giải quyết tranh chấp hợp đồng dân sự.',
    ],
}
train_path.write_text(json.dumps(record, ensure_ascii=False) + '\n', encoding='utf-8')
print(train_path.read_text(encoding='utf-8'))

## One GPU, one optimizer step

The full notebook uses two-GPU DDP. This smoke test deliberately uses one GPU because one training group cannot be distributed across two workers safely.

In [ ]:
MODEL_OUT = WORK / 'model'
!CUDA_VISIBLE_DEVICES=0 python -m FlagEmbedding.finetune.reranker.encoder_only.base \
  --model_name_or_path BAAI/bge-reranker-v2-m3 \
  --train_data {train_path} \
  --cache_path {WORK / 'cache'} \
  --train_group_size 6 \
  --query_max_len 64 \
  --passage_max_len 448 \
  --pad_to_multiple_of 8 \
  --knowledge_distillation False \
  --output_dir {MODEL_OUT} \
  --learning_rate 2e-5 \
  --fp16 \
  --max_steps 1 \
  --per_device_train_batch_size 1 \
  --gradient_accumulation_steps 1 \
  --dataloader_drop_last False \
  --gradient_checkpointing \
  --weight_decay 0.01 \
  --warmup_ratio 0.0 \
  --logging_steps 1 \
  --save_strategy no \
  --report_to none

In [ ]:
required = [MODEL_OUT / 'config.json']
weight_candidates = [MODEL_OUT / 'model.safetensors', MODEL_OUT / 'pytorch_model.bin']
if not all(path.is_file() for path in required) or not any(path.is_file() for path in weight_candidates):
    raise RuntimeError(f'Smoke training finished but model files are incomplete: {list(MODEL_OUT.iterdir())}')
print('SMOKE TEST PASSED')
print('Saved model:', MODEL_OUT)
print('Files:', sorted(path.name for path in MODEL_OUT.iterdir()))

## Level 2: two-GPU DDP, near-budget passages, and reload

Level 1 does not exercise distributed training. This second test creates two groups so each GPU receives data, uses long passages, runs one DDP optimizer step, saves the model, reloads it through Transformers, and verifies finite inference logits.

In [ ]:
if torch.cuda.device_count() < 2:
    raise RuntimeError(f'Level-2 smoke test requires two GPUs, found {torch.cuda.device_count()}')
long_positive = ' '.join(['phạm nhân được học văn hóa xóa mù chữ theo quy định'] * 45)
long_negative = ' '.join(['quy định thủ tục hành chính tài khoản doanh nghiệp'] * 50)
ddp_records = [
    {
        'query': 'Phạm nhân có được học văn hóa để xóa mù chữ không?',
        'pos': [long_positive],
        'neg': [long_negative + f' trường hợp {i}' for i in range(5)],
    },
    {
        'query': 'Việc giáo dục và học tập của phạm nhân được quy định thế nào?',
        'pos': [long_positive + ' giáo dục'],
        'neg': [long_negative + f' hồ sơ {i}' for i in range(5)],
    },
]
ddp_train_path = WORK / 'two_query_long.jsonl'
ddp_train_path.write_text(
    ''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in ddp_records),
    encoding='utf-8',
)
print('DDP smoke groups:', len(ddp_records))

In [ ]:
DDP_MODEL_OUT = WORK / 'model_ddp'
!torchrun --nproc_per_node 2 \
  -m FlagEmbedding.finetune.reranker.encoder_only.base \
  --model_name_or_path BAAI/bge-reranker-v2-m3 \
  --train_data {ddp_train_path} \
  --cache_path {WORK / 'cache_ddp'} \
  --train_group_size 6 \
  --query_max_len 64 \
  --passage_max_len 448 \
  --pad_to_multiple_of 8 \
  --knowledge_distillation False \
  --output_dir {DDP_MODEL_OUT} \
  --learning_rate 2e-5 \
  --fp16 \
  --max_steps 1 \
  --per_device_train_batch_size 1 \
  --gradient_accumulation_steps 1 \
  --dataloader_drop_last True \
  --gradient_checkpointing \
  --weight_decay 0.01 \
  --warmup_ratio 0.0 \
  --logging_steps 1 \
  --save_strategy no \
  --ddp_find_unused_parameters False \
  --report_to none

In [ ]:
import math
from transformers import AutoModelForSequenceClassification, AutoTokenizer

weights = [DDP_MODEL_OUT / 'model.safetensors', DDP_MODEL_OUT / 'pytorch_model.bin']
if not (DDP_MODEL_OUT / 'config.json').is_file() or not any(path.is_file() for path in weights):
    raise RuntimeError('DDP smoke model was not saved correctly')
reload_tokenizer = AutoTokenizer.from_pretrained(DDP_MODEL_OUT, use_fast=True)
reload_model = AutoModelForSequenceClassification.from_pretrained(DDP_MODEL_OUT).cuda().eval()
batch = reload_tokenizer(
    [ddp_records[0]['query'], ddp_records[0]['query']],
    [ddp_records[0]['pos'][0], ddp_records[0]['neg'][0]],
    padding=True, truncation='only_second', max_length=512, return_tensors='pt',
)
with torch.inference_mode():
    logits = reload_model(**{k: v.cuda() for k, v in batch.items()}).logits.view(-1).float().cpu().tolist()
if len(logits) != 2 or not all(math.isfinite(value) for value in logits):
    raise RuntimeError(f'Invalid reloaded inference logits: {logits}')
print('LEVEL-2 DDP SMOKE TEST PASSED')
print('Reloaded logits:', logits)